In [1]:
import numpy as np
from pprint import pprint

In [2]:
# fmt: off
np.set_printoptions(
    edgeitems=30, 
    linewidth=100_000, 
    precision=3, 
    suppress=True
)

## Array Creation

### Data Types

Numpy supports a bunch of **fixed-width** numeric, boolean, string, and byte data types. Some data types like `np.int_` and `np.intp` are platform specific. For fixed-width strings, any additional characters are chopped off, and extra space is padded with zeros. Here is a small list of useful data types:
  * `np.bool`
  * `np.int8` and `np.uint8`
  * `np.int16` and `np.uint16`
  * `np.int32` and `np.uint32`
  * `np.float32`
  * `np.float64`
  * `np.float128`
  * `"U"`: to specify unicode strings, this is usually followed by the size, e.g., `U5`.
  * `"S"`: to specify byte strings, also followed by the size, e.g., `S4`.

In [4]:
tp = np.array(["APTG", "Lala"], dtype="U4")
tp.dtype

dtype('<U4')

In [5]:
tp = np.array(["Chocolate Chip", "Snicker Doodle"], dtype=str)
tp.dtype

dtype('<U14')

### Types of Arrays

For the most part I'll be working with `ndarray` type. Ther is also a [`structured array`](https://numpy.org/doc/stable/user/basics.rec.html#structured-arrays) type which are like named tuples. I have never used them before but good to know that they exist.

In [6]:
# fmt: off
cookie_type = np.dtype([
    ("flavor", "U20"),
    ("calories", np.int16)
])

In [8]:
# fmt: off
cookies = np.array([
    ("Chocolate Chip", 200),
    ("Snicker Doodle", 220)
], dtype=cookie_type)
cookies

array([('Chocolate Chip', 200), ('Snicker Doodle', 220)], dtype=[('flavor', '<U20'), ('calories', '<i2')])

In [14]:
cookies[0]

np.void(('Chocolate Chip', 200), dtype=[('flavor', '<U20'), ('calories', '<i2')])

In [15]:
cookies[0]["flavor"]

np.str_('Chocolate Chip')

### Array Creation Functions

**1-D array creation functions:**
  * `np.arange(start, stop, step)`
  * `np.linspace(start, stop, num_elements)`

**2-D array creation functions:**
  * `np.eye()`
  * `np.diag()`

**N-D array creation functions:**
  * `np.zeros((...dims...)[, dtype=<type>])` and `np.zeros_like()`
  * `np.ones((...dims...)[, dtype=<type>])` and `np.ones_like()`
  * `np.empty((...dims...)[, dtype=<type>])` and `np.empty_like()`
  * `np.fromfunction(fn, shape, *, dtype, like)` will pass the coordinates of the array to the given function, and the output of the function will put in the passed in coordinates.

The default `dtype` is `np.float64`.

`arange()` can be unpredictable when generating floats, `linspace` is better suited for that.

In [16]:
np.arange(1, 5)

array([1, 2, 3, 4])

In [17]:
np.linspace(1, 5, 10)

array([1.   , 1.444, 1.889, 2.333, 2.778, 3.222, 3.667, 4.111, 4.556, 5.   ])

In [18]:
np.fromfunction(lambda i, j: 10 * i + j, (3, 5))

array([[ 0.,  1.,  2.,  3.,  4.],
       [10., 11., 12., 13., 14.],
       [20., 21., 22., 23., 24.]])

It is good to understand the params that are passed to the function in the `fromfunction()` function. My initial understanding was that each coordinate was passed to the function in an iterative manner. That is incorrect. The function is callled just once, with the coordinates passed in as arrays of the same shape as the output. In the example below, the first param varies the coordinates in the first (`0`th) axis, so the first matrix is all `0`s and the second matrix is all `1`s. The second param is varying the coordinates in the second (`1`st) axis, so the second matrix has the rows varying, the the third param is varying the third axis. However, it is good to have the old flawed mental model because it is much easier to reason about what the output will be. 

In [19]:
def fn(i, j, k):
    print(f"i = \n{i}")
    print(f"j = \n{j}")
    print(f"k = \n{k}")
    print("---")
    return i + j + k


np.fromfunction(fn, (2, 3, 4))

i = 
[[[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]]
j = 
[[[0. 0. 0. 0.]
  [1. 1. 1. 1.]
  [2. 2. 2. 2.]]

 [[0. 0. 0. 0.]
  [1. 1. 1. 1.]
  [2. 2. 2. 2.]]]
k = 
[[[0. 1. 2. 3.]
  [0. 1. 2. 3.]
  [0. 1. 2. 3.]]

 [[0. 1. 2. 3.]
  [0. 1. 2. 3.]
  [0. 1. 2. 3.]]]
---


array([[[0., 1., 2., 3.],
        [1., 2., 3., 4.],
        [2., 3., 4., 5.]],

       [[1., 2., 3., 4.],
        [2., 3., 4., 5.],
        [3., 4., 5., 6.]]])

In [20]:
def fn(i, j):
    print(f"i = \n{i}")
    print(f"j = \n{j}")
    print("---")
    return 10 * i + j


np.fromfunction(fn, (5, 4))

i = 
[[0. 0. 0. 0.]
 [1. 1. 1. 1.]
 [2. 2. 2. 2.]
 [3. 3. 3. 3.]
 [4. 4. 4. 4.]]
j = 
[[0. 1. 2. 3.]
 [0. 1. 2. 3.]
 [0. 1. 2. 3.]
 [0. 1. 2. 3.]
 [0. 1. 2. 3.]]
---


array([[ 0.,  1.,  2.,  3.],
       [10., 11., 12., 13.],
       [20., 21., 22., 23.],
       [30., 31., 32., 33.],
       [40., 41., 42., 43.]])

### Copying Arrays

Most of the time, arrays are **not** copied, different indexing, other ops, are simply a different view into the same underlying data, change one, and the original data and all other variables referencing it will change. I can explicitly copy arrays using `np.copy()` which will create a deep copy.

In [26]:
a = np.arange(2 * 3 * 4).reshape((2, 3, 4))
a

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [27]:
b = np.copy(a)
b

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [28]:
b[0, 0, 0] = -1
b

array([[[-1,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [29]:
a

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

### Random Numbers

To generate random numbers I need to use a [random number generator](https://numpy.org/doc/stable/reference/random/generator.html). Fortunately NumPy comes with a defaut random number generator. It supports a bunch of different probability distributions as can be seen from the link above. Here are the most common ones that end up using a lot:

  * `Generator.integers(low, high=None, size=None, dtype=np.int64, endpoint=False)`
  * `Generator.random(size=None, dtype=np.float64, out=None)`
  * `Generator.choice(a, size=None, replace=True, p=None, axis=0, shuffle=True)`
  * `Generator.shuffle(a, axis=0)`
  * `Generator.normal(loc=0.0, scale=1.0, size=None)`

In [ ]:
rng = np.random.default_rng()

In [ ]:
rng.integers(10, 15, 10, dtype=np.int32)